# Day 060 — Exercise 1: SimpleCache

The simplest caching strategy is an **in-memory dict** where each entry has a **time-to-live (TTL)**. After TTL seconds the entry expires and the next `get` returns `None`, forcing a fresh fetch.

Key design: `_store[key] = (value, expires_at)` where `expires_at` is `time.monotonic() + ttl`. Use `time.monotonic()` — it never goes backwards and is unaffected by system clock changes.

In [ ]:
import time
from typing import Any


## Task

Implement `SimpleCache`:

| Method | Behaviour |
|--------|-----------|
| `set(key, value, ttl=60)` | Store `(value, now + ttl)` |
| `get(key) -> Any\|None` | Return value if not expired; evict and return `None` if expired |
| `has(key) -> bool` | `True` if key exists and is not expired |
| `delete(key)` | Remove key (no-op if missing) |
| `clear() -> int` | Remove all entries, return count |
| `__len__() -> int` | Count of non-expired entries |

## Your Implementation

In [ ]:
class SimpleCache:
    """In-memory key-value cache with per-entry TTL.

    set(key, value, ttl)  — store value; expires after ttl seconds
    get(key) -> Any|None  — return value if not expired, else None
    has(key) -> bool      — True if key exists and is not expired
    delete(key)           — remove key (no-op if missing)
    clear() -> int        — remove all entries, return count removed
    __len__() -> int      — count of non-expired entries
    """

    def __init__(self):
        # TODO: init _store dict: key -> (value, expires_at)
        raise NotImplementedError

    def set(self, key: str, value: Any, ttl: float = 60.0) -> None:
        raise NotImplementedError

    def get(self, key: str) -> Any:
        raise NotImplementedError

    def has(self, key: str) -> bool:
        raise NotImplementedError

    def delete(self, key: str) -> None:
        raise NotImplementedError

    def clear(self) -> int:
        raise NotImplementedError

    def __len__(self) -> int:
        raise NotImplementedError


In [ ]:
class SimpleCache:
    def __init__(self):
        self._store: dict = {}

    def set(self, key: str, value: Any, ttl: float = 60.0) -> None:
        self._store[key] = (value, time.monotonic() + ttl)

    def get(self, key: str) -> Any:
        entry = self._store.get(key)
        if entry is None:
            return None
        value, expires_at = entry
        if time.monotonic() > expires_at:
            del self._store[key]
            return None
        return value

    def has(self, key: str) -> bool:
        return self.get(key) is not None

    def delete(self, key: str) -> None:
        self._store.pop(key, None)

    def clear(self) -> int:
        n = len(self._store)
        self._store.clear()
        return n

    def __len__(self) -> int:
        now = time.monotonic()
        return sum(1 for _, exp in self._store.values() if now <= exp)


## Automated checks

In [ ]:
score, total = 0, 6
try:
    c = SimpleCache()

    # set and get
    c.set("k1", "hello", ttl=10.0)
    assert c.get("k1") == "hello", f"Expected 'hello', got {c.get('k1')}"
    score += 1; print("\u2705 set/get stores and retrieves value")

    # has
    assert c.has("k1") is True
    assert c.has("nope") is False
    score += 1; print("\u2705 has() returns True/False correctly")

    # expired entry
    c.set("k2", "short", ttl=0.01)
    time.sleep(0.05)
    assert c.get("k2") is None, f"Expired entry should be None, got {c.get('k2')}"
    score += 1; print("\u2705 expired entry returns None")

    # delete
    c.set("k3", "val", ttl=10.0)
    c.delete("k3")
    assert c.get("k3") is None
    c.delete("no_such_key")   # should not raise
    score += 1; print("\u2705 delete removes entry (no-op for missing)")

    # __len__ counts non-expired
    c2 = SimpleCache()
    c2.set("a", 1, ttl=10.0)
    c2.set("b", 2, ttl=0.01)
    time.sleep(0.05)
    assert len(c2) == 1, f"Expected 1 non-expired entry, got {len(c2)}"
    score += 1; print("\u2705 __len__ counts only non-expired entries")

    # clear
    c3 = SimpleCache()
    c3.set("x", 1, ttl=10.0)
    c3.set("y", 2, ttl=10.0)
    removed = c3.clear()
    assert removed == 2, f"Expected clear() to return 2, got {removed}"
    assert len(c3) == 0
    score += 1; print("\u2705 clear() removes all entries and returns count")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class SimpleCache:
    def __init__(self):
        self._store: dict = {}

    def set(self, key: str, value: Any, ttl: float = 60.0) -> None:
        self._store[key] = (value, time.monotonic() + ttl)

    def get(self, key: str) -> Any:
        entry = self._store.get(key)
        if entry is None:
            return None
        value, expires_at = entry
        if time.monotonic() > expires_at:
            del self._store[key]
            return None
        return value

    def has(self, key: str) -> bool:
        return self.get(key) is not None

    def delete(self, key: str) -> None:
        self._store.pop(key, None)

    def clear(self) -> int:
        n = len(self._store)
        self._store.clear()
        return n

    def __len__(self) -> int:
        now = time.monotonic()
        return sum(1 for _, exp in self._store.values() if now <= exp)
```

**Why `time.monotonic()`?** Unlike `time.time()`, the monotonic clock never goes backwards — NTP corrections can adjust `time.time()` by seconds, which would cause entries to expire prematurely or never. `monotonic` is purely for measuring elapsed time.

</details>